# Collecting YouTube comments for all videos in a playlist

This script collects YouTube comments for all videos in a YouTube playlist, using the *YouTube Data API* of Google. The code is based on the instructions provided in the following Github repository:
https://github.com/rodolflying/youtube_automation/blob/master/main.py

Changes were made by Monika Barget in October 2024 to adapt the code for the data science research infrastructure (DSRI) at Maastricht University and to make the code easier to use for students without programming experience.
Arnoud Wils (The Plant) provided valuable feedback when we first tested the code in class.

In 2026, the script was adjusted for use in Pyodide, so that it runs directly in your browser. Instead of text files, you now type your **API key** and your **playlist URL** into input boxes, and the results can be downloaded at the end. Requests are sent directly from your browser to the YouTube Data API.

**What you need**
- a YouTube Data API key of your own (see the next section)
- the URL of a **public** YouTube playlist (private playlists cannot be read with an API key)

Then run the grey fields of this notebook one after the other from the top. Nothing is saved automatically, and everything is gone when you close the tab.

## Before you start: get a YouTube API key

An API key identifies you towards Google and lets you use the YouTube Data API for free within a daily limit. You create it yourself in the Google Cloud Console:

1. Sign in at the [Google Cloud Console](https://console.cloud.google.com/) with a Google account and create a new project (or choose an existing one).
2. In the API library, enable the **YouTube Data API v3** for this project.
3. Go to the credentials section and create an **API key**.
4. Restrict the key (under "API restrictions") to the **YouTube Data API v3** only.

The exact menu names change from time to time. Google explains the steps in its [getting started guide](https://developers.google.com/youtube/v3/getting-started).

**How much can you collect?** By default, each Google Cloud project may use 10,000 quota units per day. Every request for a page of playlist items or a page of comments costs 1 unit ([quota table](https://developers.google.com/youtube/v3/determine_quota_cost)). This notebook asks for up to 100 comments per request, so a playlist with a few thousand comments fits easily into one day.

## Your API key: what happens to it and how to protect it

An API key works like a password for your own Google quota, so please treat it as private.

**What this notebook does with your key**
- You type it into a box on this page. It is kept only in the memory of this browser tab. The notebook never prints it, never writes it to a file and never stores it anywhere else.
- Your browser sends it to **Google (the YouTube Data API) and to nobody else**. Google needs it with every request to know whose quota to use. This notebook does not send it to the course website, to your teachers or to any other server.
- When you close or reload the tab, the key is gone from memory. The last step of this notebook also removes it on request.

**What this notebook cannot control**
Your web browser, browser extensions and password managers may offer to remember or autofill what you type into web pages, and some extensions can read the content of pages. On a shared computer, other people might find such saved entries later.

**So please:**
1. Use a **private (incognito) window** for this notebook. Private windows discard form data and history when you close them, and browsers usually switch extensions off there by default.
2. Never paste your key into a code cell or a text file, and never share or save a notebook that contains it.
3. Create a key just for this course, restrict it to the YouTube Data API v3, and delete it or create a new one when you are finished.
4. Do not enter your key on a computer that you share with other people.

## Step 1: Set up

The first run can take a moment while `pandas` is loaded into your browser. Please wait for the "Done!" message before continuing!

In [ ]:
import micropip
await micropip.install("pandas")

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)  # hides the harmless pandas/pyarrow notice

import asyncio
import base64
import getpass
import io
import re
import zipfile
from urllib.parse import urlencode

import pandas as pd
from IPython.display import display, HTML
from pyodide.http import pyfetch

# Everything you enter below is kept in this dictionary, in the memory of this browser tab only.
# It is never printed, never written to a file and gone when you close the tab.
session = {}

def ask_value(key, label, secret=False, placeholder=""):
    # Shows an input box and stores what you type in session[key].
    # If input boxes cannot be shown in this environment, a text prompt is used instead.
    session.pop(key, None)
    field_type = "password" if secret else "text"
    display(HTML(
        f'<div style="margin:6px 0"><b>{label}</b><br>'
        f'<input id="yt-{key}" type="{field_type}" size="60" autocomplete="new-password" placeholder="{placeholder}"> '
        f'<button id="yt-{key}-btn">Save</button> <span id="yt-{key}-status"></span></div>'
    ))

    field = button = status = create_proxy = None
    try:
        from js import document
        from pyodide.ffi import create_proxy
        field = document.getElementById(f"yt-{key}")
        button = document.getElementById(f"yt-{key}-btn")
        status = document.getElementById(f"yt-{key}-status")
    except Exception:
        field = None

    def key_warning(value):
        if secret and not (value.startswith("AIza") and len(value) == 39):
            return " Note: Google API keys usually start with 'AIza' and have 39 characters. Please check for typos or extra spaces."
        return ""

    if field is None or button is None or status is None:
        # Fallback: ask with a text prompt (a masked prompt for the API key)
        prompt = f"{label}: "
        try:
            value = getpass.getpass(prompt) if secret else input(prompt)
        except Exception:
            value = input(prompt)
        value = value.strip()
        if value:
            session[key] = value
            print(f"{label} received." + key_warning(value))
        else:
            print(f"No {label.lower()} entered.")
        return

    def _on_click(event):
        value = field.value.strip()
        if not value:
            status.textContent = " Nothing entered yet."
            return
        session[key] = value
        if secret:
            field.value = ""  # clear the box so the key is not left on screen
            status.textContent = f" Saved ({len(value)} characters). The box has been cleared." + key_warning(value)
        else:
            status.textContent = " Saved."

    button.addEventListener("click", create_proxy(_on_click))

print("Done!")

## Step 2: Enter your API key

Run the cell below, paste your API key into the box and click **Save**. The key is hidden while you type. If no box appears, a prompt asks for the key instead: type or paste it there and press Enter.

In [ ]:
ask_value("api_key", "YouTube API key", secret=True)

## Step 3: Enter your playlist URL

Run the cell below, paste the URL of your YouTube playlist into the box and click **Save**. The URL looks like this: `https://www.youtube.com/playlist?list=PL...` (it must contain `list=`).

In [ ]:
ask_value("playlist_url", "YouTube playlist URL", placeholder="https://www.youtube.com/playlist?list=...")

## Step 4: Define the functions

Run the cell below. Nothing visible happens: it only teaches Python how to talk to the YouTube Data API.

In [ ]:
API_BASE = "https://www.googleapis.com/youtube/v3"
state = {"quota_exceeded": False}

class YouTubeAPIError(Exception):
    def __init__(self, message, reason=""):
        super().__init__(message)
        self.reason = reason

def _safe(text):
    # make sure the API key can never end up in a printed message
    key = session.get("api_key")
    return str(text).replace(key, "[API key]") if key else str(text)

async def api_get(endpoint, **params):
    # send one request from your browser to the YouTube Data API
    query = urlencode({**params, "key": session["api_key"]})
    response = await pyfetch(f"{API_BASE}/{endpoint}?{query}")
    try:
        data = await response.json()
    except Exception:
        data = {}
    if not response.ok:
        error = data.get("error", {})
        reason = (error.get("errors") or [{}])[0].get("reason", "")
        raise YouTubeAPIError(error.get("message", f"HTTP {response.status}"), reason)
    return data

def get_playlist_id(playlist_url):
    # extract the playlist ID from the playlist URL
    match = re.search(r"list=([^&]+)", playlist_url)
    if match:
        return match.group(1)
    raise ValueError("Invalid playlist URL: it should contain 'list=' followed by the playlist ID.")

async def get_video_ids_from_playlist(playlist_id, max_videos=None):
    # fetch the video IDs of all videos in the playlist
    video_ids = []
    page_token = None
    while True:
        params = {"part": "snippet", "playlistId": playlist_id, "maxResults": 50}
        if page_token:
            params["pageToken"] = page_token
        data = await api_get("playlistItems", **params)
        for item in data.get("items", []):
            video_ids.append(item["snippet"]["resourceId"]["videoId"])
        page_token = data.get("nextPageToken")
        if not page_token or (max_videos and len(video_ids) >= max_videos):
            break
    return video_ids[:max_videos] if max_videos else video_ids

async def get_comments(video_id, max_comments=None):
    # fetch the comments (and their replies) for one video
    rows = []
    page_token = None
    pages = 0
    while True:
        params = {"part": "snippet,replies", "videoId": video_id, "textFormat": "plainText", "maxResults": 100}
        if page_token:
            params["pageToken"] = page_token
        try:
            data = await api_get("commentThreads", **params)
        except YouTubeAPIError as e:
            if e.reason in ("quotaExceeded", "dailyLimitExceeded"):
                state["quota_exceeded"] = True
            print(f"   Could not collect (more) comments for this video: {_safe(e)}")
            break
        except Exception as e:
            print(f"   Error while collecting comments for this video: {_safe(e)}")
            break

        for item in data.get("items", []):
            top = item["snippet"]["topLevelComment"]["snippet"]
            replies = [r["snippet"]["textDisplay"] for r in item.get("replies", {}).get("comments", [])]
            rows.append({
                "comment": top["textDisplay"],
                "replies": replies,
                "date": top["publishedAt"],
                "user_name": top["authorDisplayName"],
            })

        pages += 1
        if pages % 10 == 0:
            print(f"   ... {len(rows)} comments so far. Please be patient!")
        page_token = data.get("nextPageToken")
        if not page_token or (max_comments and len(rows) >= max_comments):
            break
        await asyncio.sleep(2)  # short pause between requests

    if max_comments:
        rows = rows[:max_comments]
    return pd.DataFrame(rows, columns=["comment", "replies", "date", "user_name"])

print("Functions defined!")

## Step 5: Collect the comments

Run the cell below and wait for output. Videos with many comments can take a few minutes, so please be patient and keep this browser tab open. If an error occurs, let your tutor know. You should see the "All comments scraped!" message when the process has been successfully completed.

If you only want to try the notebook first, you can limit the amount of data with the two settings at the top of the cell (for example `max_videos = 3`).

In [ ]:
max_videos = None              # None = all videos; for example 3 = only the first three videos
max_comments_per_video = None  # None = all comments; for example 500 = at most 500 comments per video

comments_by_video = {}         # video ID -> table with the comments
state["quota_exceeded"] = False

if not session.get("api_key"):
    print("No API key yet. Go back to Step 2 and save your API key first.")
elif not session.get("playlist_url"):
    print("No playlist URL yet. Go back to Step 3 and save your playlist URL first.")
else:
    try:
        playlist_id = get_playlist_id(session["playlist_url"])
        video_ids = await get_video_ids_from_playlist(playlist_id, max_videos)
    except Exception as e:
        print(f"Could not read the playlist: {_safe(e)}")
        video_ids = []

    if video_ids:
        print(f"Found {len(video_ids)} video(s) in the playlist.")
        for number, video_id in enumerate(video_ids, start=1):
            print(f"Video {number} of {len(video_ids)} (ID {video_id}): collecting comments ...")
            df = await get_comments(video_id, max_comments_per_video)
            print(f"   {len(df)} comments collected.")
            if len(df) > 0:
                comments_by_video[video_id] = df
            if state["quota_exceeded"]:
                print("Your daily YouTube API quota is used up. The comments collected so far are kept. "
                      "Try again tomorrow or with another API key.")
                break
        else:
            print("All comments scraped!")

## Step 6: Preview the results

The table shows how many comments were collected per video, followed by the first rows for the first video.

In [ ]:
if not comments_by_video:
    print("No comments collected yet. Complete Steps 2 to 5 first.")
else:
    summary = pd.DataFrame({
        "video_id": list(comments_by_video.keys()),
        "comments": [len(d) for d in comments_by_video.values()],
    })
    display(summary)
    first_id = next(iter(comments_by_video))
    print(f"First rows for video {first_id}:")
    display(comments_by_video[first_id].head())

## Step 7: Download your results

Because this notebook runs inside your browser, there is no folder to write to. Running the cell below creates download links:
- one link for **all files together** as a `.zip` archive (if you collected several videos)
- one link **per video** (click "Show individual files"), named `VIDEOID_user_comments.csv` like in the original script

Click a link to save the file to your computer.

In [ ]:
def _data_link(name, data, mime, label):
    b64 = base64.b64encode(data).decode("ascii")
    return f'<div><a download="{name}" href="data:{mime};base64,{b64}">{label}</a></div>'

if not comments_by_video:
    print("Nothing to download yet. Complete Steps 2 to 5 first.")
else:
    csv_files = {
        f"{video_id}_user_comments.csv": table.to_csv(index=False, encoding="utf-8").encode("utf-8")
        for video_id, table in comments_by_video.items()
    }

    html = ""
    if len(csv_files) > 1:
        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
            for name, data in csv_files.items():
                zf.writestr(name, data)
        html += _data_link("YouTube_comments.zip", buffer.getvalue(), "application/zip",
                           f"Download all {len(csv_files)} files as YouTube_comments.zip")

    html += "<details><summary>Show individual files</summary>"
    html += "".join(_data_link(name, data, "text/csv", f"Download {name}") for name, data in csv_files.items())
    html += "</details>"
    display(HTML(html))
    print("Click the links above to save your results.")

## Step 8: Remove your API key from this session

When you are finished, run the cell below to remove your API key from the memory of this tab. Closing the tab does the same. If you want to be extra careful, also delete the key in the Google Cloud Console.

In [ ]:
session.pop("api_key", None)
print("Your API key has been removed from this session.")